# RetinaVessel AI — multi-seed replication v5 (seed 123)

This notebook independently fits the standard U-Net, clean lightweight U-Net and degradation-trained lightweight U-Net with training RNG seed 123. The audited image split remains frozen. After training, it executes the complete validation-only v3 robustness/topology matrix.

**Protocol boundary:** seed 123 was predeclared with seeds 7 and 42. Do not alter model settings based on seed-42 or seed-7 results, and do not inspect or evaluate the official FIVES test images. Enable a GPU accelerator.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/simlexx-k/retinavessel-ai.git"
EXPECTED_COMMIT = "224e115808a3330ee17516f0348ccd8fda9aa5ba"
REPO = Path("/kaggle/working/retinavessel-ai-v5-seed123-code")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_commit == EXPECTED_COMMIT
os.chdir(REPO)
print("Pinned replication code:", actual_commit)

Cloning into '/kaggle/working/retinavessel-ai-v5-seed123-code'...
HEAD is now at 224e115 Fix backend imports in Kaggle v3 test cell


Pinned replication code: 224e115808a3330ee17516f0348ccd8fda9aa5ba


In [2]:
import torch

sys.path.insert(0, str(REPO / "src"))
from retinavessel.data import find_fives_root, load_manifest, samples_from_frozen_split

assert torch.cuda.is_available(), "Enable a GPU accelerator before running v5"
DATA_MOUNT = Path("/kaggle/input/datasets/simonkosgei/fives-retinal-dataset")
assert DATA_MOUNT.exists(), f"Missing attached FIVES dataset: {DATA_MOUNT}"
FIVES_ROOT = find_fives_root(DATA_MOUNT)
SPLIT_MANIFEST = REPO / "splits/fives_seed42.json"
EXPECTED_SPLIT_SHA256 = "aee8809176974d6f880013d85a3ab4a405a43121d0cf31363fcc47dd75256c87"
assert hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest() == EXPECTED_SPLIT_SHA256
split = json.loads(SPLIT_MANIFEST.read_text())
train_samples, validation_samples = samples_from_frozen_split(load_manifest(FIVES_ROOT, "train"), split)
assert len(train_samples) == 504 and len(validation_samples) == 89
assert len(load_manifest(FIVES_ROOT, "test")) == 200  # count only; no pixels opened
print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Frozen split: 504 train / 89 validation / 7 excluded")
print("Official test images inspected or evaluated: 0")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

PyTorch: 2.11.0+cu130
CUDA device: Tesla T4
Frozen split: 504 train / 89 validation / 7 excluded
Official test images inspected or evaluated: 0
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], returncode=0)

In [3]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), str(REPO / "backend")])
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, env=test_environment, check=True)

..............                                                           [100%]
14 passed in 9.22s


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

In [4]:
MAX_EPOCHS = 30
PATIENCE = 5
SEED = 123
RUN_TIMES = {}
EXPERIMENTS = [
    {"name": "unet_seed123_es", "model": "unet", "degradation": False},
    {"name": "lite_unet_seed123_es", "model": "lite_unet", "degradation": False},
    {"name": "lite_robust_seed123_es", "model": "lite_unet", "degradation": True},
]

def run_experiment(spec):
    output = REPO / "outputs" / spec["name"]
    last_checkpoint = output / "last.pt"
    command = [
        sys.executable, "-m", "retinavessel.train",
        "--data", str(DATA_MOUNT), "--model", spec["model"],
        "--epochs", str(MAX_EPOCHS), "--patience", str(PATIENCE),
        "--batch-size", "2", "--patch-size", "384",
        "--patches-per-image", "2", "--overlap", "64",
        "--split-manifest", str(SPLIT_MANIFEST),
        "--seed", str(SEED), "--device", "cuda",
        "--out", str(output),
    ]
    if spec["degradation"]:
        command.append("--degradation")
    if last_checkpoint.exists():
        state = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
        complete = int(state["epoch"]) >= MAX_EPOCHS or int(state["epochs_without_improvement"]) >= PATIENCE
        if complete:
            print(f"{spec['name']} already complete at epoch {state['epoch']}; skipping")
            return
        command.extend(["--resume", str(last_checkpoint)])
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(REPO / "src")
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO, env=environment, check=True)
    RUN_TIMES[spec["name"]] = time.perf_counter() - started
    for required in ("best.pt", "last.pt", "history.json", "split_manifest.json"):
        assert (output / required).is_file(), f"Missing {output / required}"

def save_partial_snapshot(label):
    target = Path(f"/kaggle/working/retinavessel_v5_seed123_{label}")
    archive = Path(shutil.make_archive(str(target), "zip", root_dir=REPO, base_dir="outputs"))
    print("Partial recovery archive:", archive)

## E1 replication — standard U-Net, seed 123

In [5]:
run_experiment(EXPERIMENTS[0])
save_partial_snapshot("after_unet")

device=cuda, model=unet, params=1,964,097


{'epoch': 1, 'training_loss': 0.9527246723217624, 'validation_dice': 0.3885835946263992}


{'epoch': 2, 'training_loss': 0.7512753698087874, 'validation_dice': 0.3957619047574264}


{'epoch': 3, 'training_loss': 0.6974682991230299, 'validation_dice': 0.38369913215708007}


{'epoch': 4, 'training_loss': 0.6412241113564324, 'validation_dice': 0.4722356396832541}


{'epoch': 5, 'training_loss': 0.6060067944169517, 'validation_dice': 0.47407944864140444}


... (21 outputs left out)

{'epoch': 27, 'training_loss': 0.33484594609647517, 'validation_dice': 0.8065937856854428}


{'epoch': 28, 'training_loss': 0.3224525927078156, 'validation_dice': 0.7902261696855584}


{'epoch': 29, 'training_loss': 0.3204504539155298, 'validation_dice': 0.7948147398990776}


{'epoch': 30, 'training_loss': 0.31699723272865254, 'validation_dice': 0.7984610935165503}
early stopping at epoch 30; best validation Dice=0.817998
Partial recovery archive: /kaggle/working/retinavessel_v5_seed123_after_unet.zip


## E2 replication — clean lightweight U-Net, seed 123

In [6]:
run_experiment(EXPERIMENTS[1])
save_partial_snapshot("after_lite")

device=cuda, model=lite_unet, params=236,156


{'epoch': 1, 'training_loss': 0.9742038619541933, 'validation_dice': 0.6632264956025309}


{'epoch': 2, 'training_loss': 0.49851723322792657, 'validation_dice': 0.7482431463445985}


{'epoch': 3, 'training_loss': 0.4336879882399762, 'validation_dice': 0.7499474413373604}


{'epoch': 4, 'training_loss': 0.38588395676324294, 'validation_dice': 0.7574752939851798}


{'epoch': 5, 'training_loss': 0.3755297729715941, 'validation_dice': 0.782678625845002}


... (3 outputs left out)

{'epoch': 9, 'training_loss': 0.3438331715112168, 'validation_dice': 0.7703423419361093}


{'epoch': 10, 'training_loss': 0.3226661009654876, 'validation_dice': 0.8141756479773035}


{'epoch': 11, 'training_loss': 0.3274284175316256, 'validation_dice': 0.8018441296866066}


{'epoch': 12, 'training_loss': 0.3256915689017328, 'validation_dice': 0.8113238495793015}
early stopping at epoch 12; best validation Dice=0.815670
Partial recovery archive: /kaggle/working/retinavessel_v5_seed123_after_lite.zip


## E3 replication — degradation-trained lightweight U-Net, seed 123

In [7]:
run_experiment(EXPERIMENTS[2])
save_partial_snapshot("training_complete")

device=cuda, model=lite_unet, params=236,156


{'epoch': 1, 'training_loss': 0.9820612307697062, 'validation_dice': 0.6281050848109123}


{'epoch': 2, 'training_loss': 0.5174275101532065, 'validation_dice': 0.7415688018884533}


{'epoch': 3, 'training_loss': 0.4469087135105852, 'validation_dice': 0.7509515338978487}


{'epoch': 4, 'training_loss': 0.41659452786876094, 'validation_dice': 0.7759290951197306}


{'epoch': 5, 'training_loss': 0.4012254248447125, 'validation_dice': 0.8162904617198136}


... (13 outputs left out)

{'epoch': 19, 'training_loss': 0.3062721376292526, 'validation_dice': 0.8033770197953717}


{'epoch': 20, 'training_loss': 0.32532652466010953, 'validation_dice': 0.8182601834075602}


{'epoch': 21, 'training_loss': 0.31291169451460954, 'validation_dice': 0.825105180502603}


{'epoch': 22, 'training_loss': 0.322239371829681, 'validation_dice': 0.8349349149360846}
early stopping at epoch 22; best validation Dice=0.849298
Partial recovery archive: /kaggle/working/retinavessel_v5_seed123_training_complete.zip


In [8]:
import pandas as pd
from retinavessel.model import UNet, model_params

rows = []
for spec in EXPERIMENTS:
    output = REPO / "outputs" / spec["name"]
    history = json.loads((output / "history.json").read_text())
    checkpoint_path = output / "best.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert checkpoint["seed"] == SEED
    network = UNet(base=int(checkpoint["base"]), lite=checkpoint["architecture"] == "lite_unet")
    network.load_state_dict(checkpoint["model_state"], strict=True)
    rows.append({
        "experiment": spec["name"], "model": checkpoint["architecture"],
        "degradation_training": spec["degradation"], "seed": SEED,
        "parameters": model_params(network), "completed_epochs": history[-1]["epoch"],
        "best_epoch": checkpoint["epoch"], "best_validation_dice": checkpoint["validation_dice"],
        "checkpoint_sha256": hashlib.sha256(checkpoint_path.read_bytes()).hexdigest(),
        "wall_minutes_this_notebook": round(RUN_TIMES.get(spec["name"], float("nan")) / 60, 2),
    })
summary = pd.DataFrame(rows).sort_values("best_validation_dice", ascending=False)
display(summary)
SUMMARY_DIR = REPO / "outputs" / "v5_seed123_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(SUMMARY_DIR / "validation_summary.csv", index=False)
run_manifest = {
    "training_code_commit": EXPECTED_COMMIT, "split_manifest_sha256": EXPECTED_SPLIT_SHA256,
    "pytorch_version": torch.__version__, "cuda_device": torch.cuda.get_device_name(0),
    "maximum_epochs": MAX_EPOCHS, "early_stopping_patience": PATIENCE,
    "seed": SEED, "official_test_evaluated": False,
    "results": json.loads(summary.to_json(orient="records")),
}
(SUMMARY_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))

,experiment,model,degradation_training,seed,parameters,completed_epochs,best_epoch,best_validation_dice,checkpoint_sha256,wall_minutes_this_notebook
2,lite_robust_seed123_es,lite_unet,True,123,236156,22,17,0.849298,041f0b42954da36b02812cd97738d33ee14554364cc90a...,123.10
0,unet_seed123_es,unet,False,123,1964097,30,25,0.817998,8dd71c3539ddf082b6b14c0c690321036a98557b531acf...,201.01
1,lite_unet_seed123_es,lite_unet,False,123,236156,12,7,0.815670,88b511112058089106c5ededa509dbc4528354b581d717...,72.61


1563

## Full seed-123 validation matrix

This is the same 3 × 9 × 89 validation-only matrix used for seed 42. It performs no model selection and never accesses official-test pixels.

In [9]:
VALIDATION_OUTPUT = REPO / "outputs" / "validation_v5_seed123"
evaluation_command = [
    sys.executable, "-m", "retinavessel.validation",
    "--data", str(DATA_MOUNT), "--split-manifest", str(SPLIT_MANIFEST),
    "--expected-split-sha256", EXPECTED_SPLIT_SHA256,
    "--model", f"unet_seed123={REPO / 'outputs/unet_seed123_es/best.pt'}",
    "--model", f"lite_unet_seed123={REPO / 'outputs/lite_unet_seed123_es/best.pt'}",
    "--model", f"lite_robust_seed123={REPO / 'outputs/lite_robust_seed123_es/best.pt'}",
    "--topology", "--device", "cuda", "--warmup-runs", "2",
    "--degradation-seed", "2026", "--bootstrap-seed", "2026",
    "--bootstrap-repetitions", "2000", "--out", str(VALIDATION_OUTPUT),
]
evaluation_environment = os.environ.copy()
evaluation_environment["PYTHONPATH"] = str(REPO / "src")
subprocess.run(evaluation_command, cwd=REPO, env=evaluation_environment, check=True)
report = json.loads((VALIDATION_OUTPUT / "validation_report.json").read_text())
per_image = pd.read_csv(VALIDATION_OUTPUT / "validation_per_image.csv")
assert report["official_test_evaluated"] is False
assert report["validation_image_count"] == 89 and report["debug_validation_limit"] is None
assert len(per_image) == 3 * 9 * 89
display(pd.read_csv(VALIDATION_OUTPUT / "validation_summary.csv"))

model=unet_seed123, condition=clean, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed123, condition=blur_mild, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed123, condition=blur_severe, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed123, condition=contrast_mild, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed123, condition=contrast_severe, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89

,model_id,condition,category,severity,image_count,parameters,parameter_bytes,mean_latency_ms,median_latency_ms,p95_latency_ms,...,cldice_ci95_lower,cldice_ci95_upper,mean_thin_centerline_recall,median_thin_centerline_recall,thin_centerline_recall_ci95_lower,thin_centerline_recall_ci95_upper,mean_skeleton_component_ratio,median_skeleton_component_ratio,skeleton_component_ratio_ci95_lower,skeleton_component_ratio_ci95_upper
0,unet_seed123,clean,none,clean,89,1964097,7856388,840.340771,842.465531,869.197761,...,0.798313,0.854486,0.212691,0.206146,0.185767,0.240200,22.485304,14.500000,18.338472,26.961424
1,unet_seed123,blur_mild,blur,mild,89,1964097,7856388,838.027111,837.679275,874.233267,...,0.795683,0.852227,0.213891,0.207426,0.187387,0.242299,22.731715,15.000000,18.936718,27.082581
2,unet_seed123,blur_severe,blur,severe,89,1964097,7856388,842.593174,839.356374,872.902610,...,0.789125,0.843606,0.208850,0.187970,0.182472,0.237818,23.142455,15.333333,18.909711,27.270121
3,unet_seed123,contrast_mild,contrast,mild,89,1964097,7856388,841.439598,843.167576,867.427298,...,0.748474,0.809944,0.156324,0.147807,0.136337,0.176066,25.009924,17.250000,20.930383,29.298693
4,unet_seed123,contrast_severe,contrast,severe,89,1964097,7856388,842.944062,841.133721,867.855774,...,0.649914,0.716983,0.094191,0.089251,0.081980,0.108137,30.276433,22.333333,25.310786,35.224418
5,unet_seed123,illumination_mild,illumination,mild,89,1964097,7856388,843.924321,846.075781,871.016468,...,0.737454,0.805592,0.145690,0.139501,0.128010,0.163728,22.115094,15.000000,18.260224,26.078389
6,unet_seed123,illumination_severe,illumination,severe,89,1964097,7856388,840.677419,843.917611,865.876049,...,0.617752,0.695940,0.071113,0.059365,0.059914,0.083288,23.519534,16.500000,19.466480,27.734201
7,unet_seed123,noise_mild,noise,mild,89,1964097,7856388,842.588014,844.251966,866.097528,...,0.767346,0.825557,0.144456,0.148536,0.127611,0.162272,23.143441,15.750000,18.825580,27.985560
8,unet_seed123,noise_severe,noise,severe,89,1964097,7856388,839.249633,839.796233,864.109774,...,0.676873,0.738919,0.074172,0.061069,0.063021,0.085531,23.436766,16.000000,19.081069,28.283055
9,lite_unet_seed123,clean,none,clean,89,236156,944624,457.178934,455.562879,481.958758,...,0.792384,0.848541,0.256401,0.235501,0.228043,0.286657,26.520710,16.666667,21.274756,32.180464


In [10]:
ARCHIVE_BASE = Path("/kaggle/working/retinavessel_v5_seed123")
archive = Path(shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=REPO, base_dir="outputs"))
print(f"Final artifact archive: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)")
print("Save this notebook version and download the archive before aggregation.")

Final artifact archive: /kaggle/working/retinavessel_v5_seed123.zip (35.0 MiB)
Save this notebook version and download the archive before aggregation.


## Stop here

Do not open the official FIVES test partition. Download and verify the seed-123 archive, then aggregate all three predeclared seeds. Only after aggregation may the model configuration and final test policy be frozen.